In [ ]:
from RMtools_3D import do_RMsynth_3D
from RMtools_3D import do_RMclean_3D
from RMtools_3D import do_fitIcube
from RMtools_1D import do_RMsynth_1D
import importlib
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import warnings
#warnings.filterwarnings("ignore")
import seaborn as sns
import matplotlib.gridspec as gridspec
from scipy.stats import rayleigh

### Reload RM tools in case make changes to Nergis's version. Also check we are in the right environment

In [ ]:
from importlib import reload
reload(do_RMsynth_3D)
reload(do_RMsynth_1D)
print("Module location:", do_RMsynth_3D.__file__)
print("Module name:", do_RMsynth_3D.__name__)
print("Module location:", do_RMsynth_1D.__file__)
print("Module name:", do_RMsynth_1D.__name__)

## (1) Make DRAGONS frequency sub-band (to match CHIME or other)

### Make a new directories if needed

In [ ]:
base_dir = '/srv/data/dragons/'
new_dir  = 'chime_compare_400_729'

!{'mkdir '+base_dir+new_dir}
!{'mkdir '+base_dir+new_dir+'/frequency_cubes'}
!{'mkdir '+base_dir+new_dir+'/faraday_synthesis'}
!{'mkdir '+base_dir+new_dir+'/rmsynth_coarse'}
!{'ls -lh '+base_dir}
!{'ls -lh '+base_dir+new_dir}

### Function for slicing the data

In [ ]:
def make_subband(dir_in, files_in, dir_out, fmin, fmax):

    for file_in in files_in:
        
        print(file_in)
        hdu  = fits.open(dir_in + file_in)
        hdr  = hdu[0].header
        data = hdu[0].data
        freq_all = WCS(hdr).all_pix2world(0,0,range(data.shape[0]),0)[2]
        print(data.shape)
        print(freq_all.shape)

        fidx = np.where((freq_all >= fmin) & (freq_all <= fmax))

        data_new = data[fidx]     
        hdr_new = hdr.copy()
        hdr_new['CRVAL3'] = freq_all[fidx[0][0]]/1e6
        print(data_new.shape)
        print('')

        fits.writeto(dir_out+file_in, data_new, hdr_new, overwrite=False)

    return

### Slice the data and save new files

In [ ]:
dir_in  = '/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/'
dir_out = '/srv/data/dragons/'+new_dir+'/frequency_cubes/'
fmin = 400e6
fmax = 729e6

files_in = ['dragons_freqK_gal_conv_i.car.fits',
            'dragons_freqK_gal_conv_q.car.fits',
            'dragons_freqK_gal_conv_u.car.fits',
            'dragons_freq_dQU_gal.car.fits']

make_subband(dir_in, files_in, dir_out, fmin, fmax)

## (2) Coarse RM synthesis for getting Rayleigh noise

In [ ]:
dir_in = '/srv/data/dragons/'+new_dir+'/frequency_cubes/'

hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_q.car.fits')
q_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_u.car.fits')
u_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_i.car.fits')
i_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freq_dQU_gal.car.fits')
qu_err = hdu[0].data

hdr = hdu[0].header.copy()

freqs = WCS(hdr).all_pix2world(0,0,range(q_data.shape[0]),0)[2]
print(len(freqs))
print(q_data.shape)
print(u_data.shape)
print(i_data.shape)
print(qu_err.shape)

In [ ]:
dir_out = '/srv/data/dragons/'+new_dir+'/rmsynth_coarse/'

modelI = do_fitIcube.make_model_I(i_data, hdr, freqs, outDir=dir_out, prefixOut='StokesI_', 
                                  threshold=-3, verbose=True, num_cores=16, polyOrd=5)


In [ ]:
result, freq0_Hz, lam0Sq_m2, weightArr = do_RMsynth_3D.run_rmsynth(q_data, u_data, freqs, fitRMSF=True, rmsArr=qu_err, dataI=modelI,
                                   weightType="variance", phiMax_radm2=1500, dPhi_radm2=3, verbose=True)

do_RMsynth_3D.writefits(result, hdr, write_seperate_FDF=True, outDir=dir_out)

np.savez(dir_out+'rmsynth_params', freq0_Hz=freq0_Hz, lam0Sq_m2=lam0Sq_m2, weightArr=weightArr)

## (3) Fine RM synthesis

In [ ]:
dir_out = '/srv/data/dragons/'+new_dir+'/faraday_synthesis/'

modelI = do_fitIcube.make_model_I(i_data, hdr, freqs, outDir=dir_out, prefixOut='StokesI_', 
                                  threshold=-3, verbose=True, num_cores=16, polyOrd=5)

In [ ]:
result, freq0_Hz, lam0Sq_m2, weightArr = do_RMsynth_3D.run_rmsynth(q_data, u_data, freqs, fitRMSF=True, rmsArr=qu_err, dataI=modelI,
                                   weightType="variance", phiMax_radm2=200, dPhi_radm2=0.5, verbose=True)

do_RMsynth_3D.writefits(result, hdr, write_seperate_FDF=True, outDir=dir_out)

np.savez(dir_out+'rmsynth_params', freq0_Hz=freq0_Hz, lam0Sq_m2=lam0Sq_m2, weightArr=weightArr)

## (4) Get the Rayleigh noise

In [ ]:
dir_in = '/srv/data/dragons/'+new_dir+'/rmsynth_coarse/'

dirtycube   = fits.getdata(dir_in + "FDF_tot_dirty.fits")
dirtyheader = fits.getheader(dir_in + "FDF_tot_dirty.fits")

wcs = WCS(dirtyheader)
_, __, FDvals = wcs.all_pix2world(1,1,np.arange(dirtycube.shape[0]),0)

print(FDvals)
print(wcs)

hdr2D = wcs.dropaxis(2).to_header()
print(repr(hdr2D))

In [ ]:
# Only fit the Rayleigh distribution to |phi| > threshold
thresh = 500
inds = np.abs(FDvals) > thresh

# Blank map to be populated with sigma values
sigmaArray = np.zeros([dirtycube.shape[1], dirtycube.shape[2]])

for m in range(dirtycube.shape[1]):
    print(m)
    for n in range(dirtycube.shape[2]):

        spec = dirtycube[:, m, n]

        if np.sum( np.isnan(spec) ) > 1:
            scale = np.nan
        else:
            loc, scale = rayleigh.fit(spec[inds])
        sigmaArray[m,n] = scale

fits.writeto(dir_in + 'rayleigh_FD_sigma_'+str(thresh)+'_1500.fits', sigmaArray, hdr2D, overwrite=False)
plt.imshow(sigmaArray,origin='lower',vmin=0,vmax=0.02)

In [ ]:
plt.hist(sigmaArray.flatten(),bins=100, range=(0,0.1));
print(np.nanmedian(sigmaArray.flatten()))

In [ ]:
fits.writeto(dir_in + 'five_sigma.fits',  sigmaArray*5, hdr2D, overwrite=False)

## (4) RM CLEAN

In [ ]:
dir_in_coarse = '/srv/data/dragons/'+new_dir+'/rmsynth_coarse/'
dir_in_fine   = '/srv/data/dragons/'+new_dir+'/faraday_synthesis/'

In [ ]:
cleanFDF, ccArr, iterCountArr, residFDF, head = do_RMclean_3D.run_rmclean(dir_in_fine+'FDF_tot_dirty.fits',
                                                                          dir_in_fine+'RMSF_tot.fits',
                                                                          dir_in_coarse+'five_sigma.fits', verbose=True)

In [ ]:
do_RMclean_3D.writefits(cleanFDF, ccArr, iterCountArr, residFDF, head,
                         outDir=dir_in_fine, write_separate_FDF=True)

In [ ]:
dir_in = '/srv/data/dragons/'+new_dir+'/frequency_cubes/'

hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_q.car.fits')
q_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_u.car.fits')
u_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freqK_gal_conv_i.car.fits')
i_data = hdu[0].data
hdu    = fits.open(dir_in + 'dragons_freq_dQU_gal.car.fits')
qu_err = hdu[0].data

hdr = hdu[0].header.copy()

freqs = WCS(hdr).all_pix2world(0,0,range(q_data.shape[0]),0)[2]
print(len(freqs))
print(q_data.shape)
print(u_data.shape)
print(i_data.shape)
print(qu_err.shape)

In [ ]:
hdr

In [ ]:
help(do_RMsynth_1D)

In [ ]:
def run_rm_synth(idx):

    mDict, aDict = run_rmsynth([freq_Hz, I, Q, U, dI, dQ, dU], 
                                polyOrd=5, 
                                phiMax_radm2=1500, dPhi_radm2=3, 
                                weightType='variance', 
                                fitRMSF=True, 
                                noStokesI=False)
    print(mDict)
    print(aDict)

    return

In [ ]:
freqs